# MP03 — Press Release to Plot: Industry Comparison
**CIS 3120 — Programming for Analytics · Baruch College, Zicklin School of Business**

| Role | Member |
|---|---|
| Financial Services Pipeline Lead | *Adrian D.* |
| Travel & Hospitality Pipeline Lead | *Amalie M.* |
| Comparison & Visualization Lead (Integrator) | *Sarah H.* |

**Team Number:** `12` — update before submission.

---
## 0. Setup & Dependencies
**Two manual steps required before running:**
1. Add your Anthropic API key to Colab Secrets under the name `ANTHROPIC_API_KEY`.
2. Replace the `USER_AGENT` placeholder string below with your actual name/email.

In [ ]:
# Install required packages
!pip install -q anthropic requests folium geopy pandas

In [ ]:
import os
import json
import time
import requests
import pandas as pd
import folium
from datetime import date, timedelta
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import anthropic

# ── API Key (Colab Secrets) ──────────────────────────────────────────────────
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    # Fallback: set env var manually if not running in Colab
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

assert ANTHROPIC_API_KEY, "❌ ANTHROPIC_API_KEY not found — add it to Colab Secrets."

# ── User-Agent (EDGAR requires a descriptive User-Agent) ─────────────────────
# Replace the placeholder with your real name and email before running.
USER_AGENT = "Adrian D. Adrian.Davis@baruch.cuny.edu CIS3120 MP03"

# ── Anthropic client ─────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("✅ Setup complete.")

---
## 1. Industry Ticker Lists & Search Phrases

The seeds below are the instructor-provided defaults. Both lists have been extended with justified additions documented in the Methodology section (Section 6).

In [ ]:
# ── Financial Services Tickers ───────────────────────────────────────────────
# Seed: 14 companies across money-center banks, regional banks, asset mgmt,
# insurance, and payments.
# Extensions: Added capital-markets (GS, MS), fintech (SQ, PYPL), and an
# additional regional bank (RF) to broaden geographic coverage and capture
# capital-markets location events underrepresented in the retail-banking seed.
FINANCIAL_SERVICES_TICKERS = [
    # Money-center banks
    "JPM", "BAC", "WFC", "C",
    # Capital markets (extension — seed was biased toward retail banking)
    "GS", "MS",
    # Regional banks
    "PNC", "USB", "TFC", "RF",
    # Asset management
    "BLK", "BX",
    # Insurance
    "MET", "PRU",
    # Payments / Fintech (extension — growing footprint of physical ops centers)
    "V", "MA", "AXP", "SQ", "PYPL",
]

# ── Financial Services Search Phrases ────────────────────────────────────────
# Extensions: Added capital-markets-specific phrases to capture trading floor
# and advisory office moves, and "technology center" for the wave of bank tech
# hub openings.
FINANCIAL_SERVICES_PHRASES = [
    '"new branch"',
    '"branch opening"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"regional office"',
    '"office closure"',
    '"operations center"',
    '"data center"',
    '"new location"',
    # Extensions
    '"technology center"',
    '"trading floor"',
    '"advisory office"',
    '"wealth management office"',
]

# ── Travel & Hospitality Tickers ─────────────────────────────────────────────
# Seed: 14 companies across hotels, cruise, airlines, and online travel.
# Extensions: Added resort/casino operators (MGM, WYNN, LVS) and a budget
# airline (JBLU) to capture leisure-demand expansion events beyond the
# major-carrier / major-chain seed.
TRAVEL_HOSPITALITY_TICKERS = [
    # Hotels
    "MAR", "HLT", "H", "CHH", "WH",
    # Cruise
    "CCL", "RCL", "NCLH",
    # Airlines
    "DAL", "UAL", "AAL", "LUV",
    # Budget / regional airlines (extension)
    "JBLU",
    # Online travel
    "BKNG", "EXPE",
    # Resort / Casino operators (extension — major physical-expansion filers)
    "MGM", "WYNN", "LVS",
]

# ── Travel & Hospitality Search Phrases ──────────────────────────────────────
# Extensions: Added cruise-port and casino-specific terms, and separated
# hotel-style from airline-style phrases to improve Stage 3 precision.
TRAVEL_HOSPITALITY_PHRASES = [
    '"new property"',
    '"new hotel"',
    '"hotel opening"',
    '"resort opening"',
    '"property opening"',
    '"brand conversion"',
    '"new route"',
    '"new gateway"',
    '"new terminal"',
    '"grand opening"',
    # Extensions
    '"new destination"',
    '"casino opening"',
    '"homeport"',
    '"new port"',
    '"resort expansion"',
]

print(f"Financial Services: {len(FINANCIAL_SERVICES_TICKERS)} tickers, {len(FINANCIAL_SERVICES_PHRASES)} phrases")
print(f"Travel & Hospitality: {len(TRAVEL_HOSPITALITY_TICKERS)} tickers, {len(TRAVEL_HOSPITALITY_PHRASES)} phrases")